<a href="https://colab.research.google.com/github/almagomen/v2v/blob/main/colab_launch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V2V Professional – Despliegue en Colab

**Nueva versión con Editor de Timeline Profesional**

1. **Activa GPU**: *Runtime → Change runtime type → T4 GPU*
2. Ejecuta las celdas en orden.
3. Configura **Secrets** (panel izquierdo 🔑): `GOOGLE_API_KEY`, `LMNT_API_KEY`, `AUTH0_COOKIE`, `MIXPANEL_COOKIE`

## 1. Clonar repo e instalar dependencias

In [ ]:
import os
import sys
import importlib
import warnings
import subprocess
from google.colab import drive, userdata, output
warnings.filterwarnings("ignore", category=UserWarning)

if 'initial_setup' not in globals():
    drive.mount("/content/drive")

# Gestión del Repositorio (Clone o Pull)
repo_path = "/content/v2v"
get_ghp = userdata.get('GHP_GITHUB_TOKEN')
repo_url = f"https://{get_ghp}@github.com/almagomen/v2v.git"

if not os.path.exists(repo_path):
    !git clone -q {repo_url} {repo_path}
    %cd {repo_path}
else:
    %cd {repo_path}
    !git pull

# Instalación de Dependencias Backend
if 'deps_installed' not in globals():
    print("📦 Instalando dependencias backend...")
    !uv pip install "yt-dlp[default]" bs-roformer-infer -q
    !uv pip install -U git+https://github.com/jianfch/stable-ts.git --no-deps -q
    !uv pip install openai-whisper --no-deps -q
    !uv pip install git+https://github.com/pyannote/pyannote-audio.git --no-deps -q
    !uv pip install 'pyannote-core>=6.0.1' 'pyannote-database>=6.1.1' 'pyannote-metrics>=4.0.0' 'pyannote-pipeline>=4.0.0' 'pyannoteai-sdk>=0.3.0' --no-deps -q
    !uv pip install 'opentelemetry-api>=1.34.0' 'opentelemetry-sdk>=1.34.0' 'opentelemetry-exporter-otlp>=1.34.0' sentencex 'asteroid-filterbanks>=0.4.0' 'lightning>=2.4' 'pytorch-metric-learning>=2.8.1' 'torch-audiomentations>=0.12.0' torchcodec==0.7.0 'torchmetrics>=1.6.1' 'optuna>=4.2.0' -q
    # Dependencias FastAPI
    !uv pip install fastapi uvicorn python-multipart sqlmodel aiosqlite websockets ffmpeg-python pydantic pydantic-settings -q
    deps_installed = True

print("✅ Dependencias instaladas")

## 2. Instalar y configurar Frontend (React + Vite)

In [ ]:
# Instalar Node.js y frontend
!uv pip install nodejs npm -q

frontend_path = "/content/v2v/V2V-Professional/frontend"
os.chdir(frontend_path)

# Instalar dependencias npm
!npm install 2>/dev/null

print("✅ Frontend configurado")

## 3. Configurar variables de entorno

In [ ]:
# Configurar variables de entorno para el backend
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
os.environ['LMNT_API_KEY'] = userdata.get('LMNT_API_KEY')
os.environ['LMNT_AUTH0_COOKIE'] = userdata.get('AUTH0_COOKIE')
os.environ['MIXPANEL_COOKIE'] = userdata.get('MIXPANEL_COOKIE')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['V2V_BASE_DIR'] = '/content/v2v/V2V-Professional/data'
os.environ['DEVICE'] = 'cuda'

# Crear directorios necesarios
os.makedirs('/content/v2v/V2V-Professional/data/queue', exist_ok=True)
os.makedirs('/content/v2v/V2V-Professional/data/temp', exist_ok=True)
os.makedirs('/content/v2v/V2V-Professional/data/output', exist_ok=True)

print("✅ Variables de entorno configuradas")

## 4. Cargar modelos y lanzar servicios

In [ ]:
# Importar módulos y cargar modelos
sys.path.insert(0, '/content/v2v')

from src.stt import get_whisper_model
from src.diarization import get_diarization_model
from src.download import get_roformer_model
from src.config import QUEUE_DIR, TEMP_DIR

if 'initial_setup' not in globals():
    print("🛠️  Cargando modelos...")
    get_whisper_model()
    print("   ✅ Whisper cargado")
    get_diarization_model()
    print("   ✅ Diarización cargado")
    get_roformer_model()
    print("   ✅ BS-RoFormer cargado")

initial_setup = True

print("\n✅ Modelos cargados")

## 5. Lanzar Backend y Frontend

In [ ]:
# Lanzar Backend FastAPI en background
import threading
import time

def start_backend():
    os.chdir('/content/v2v/V2V-Professional/backend')
    import uvicorn
    from app.main import app
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level="info")

backend_thread = threading.Thread(target=start_backend, daemon=True)
backend_thread.start()
time.sleep(3)

print("🚀 Backend FastAPI iniciado en puerto 8000")

In [ ]:
# Lanzar Frontend Vite en background
import threading
import time

def start_frontend():
    os.chdir('/content/v2v/V2V-Professional/frontend')
    subprocess.Popen(['npm', 'run', 'dev'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

frontend_thread = threading.Thread(target=start_frontend, daemon=True)
frontend_thread.start()
time.sleep(5)

print("🚀 Frontend Vite iniciado en puerto 3000")

In [ ]:
# Verificar que los servicios estén activos
import requests

try:
    r = requests.get('http://localhost:8000/health', timeout=5)
    print(f"✅ Backend: {r.json()}")
except Exception as e:
    print(f"⚠️ Backend: {e}")

print("\n🎉 V2V Professional listo!")
print("=" * 50)
print("📱 Frontend: http://localhost:3000")
print("🔧 Backend API: http://localhost:8000")
print("📚 API Docs: http://localhost:8000/docs")
print("=" * 50)

## 🌐 Acceder a la aplicación

Copia y pega la URL del frontend en tu navegador:

`http://localhost:3000`